# 01. Medallion pipeline: Bronze → Silver → Gold

PySpark on **Databricks**, storing every layer as **Delta** tables in Unity Catalog (`workspace.bronze`, `workspace.silver`, `workspace.gold`).

| Layer | Contents |
|---|---|
| **Bronze** | the 9 raw CSVs as they arrived, every column as text, plus load time and source file |
| **Silver** | cleaned, typed, deduplicated tables with consistent keys |
| **Gold** | small aggregated tables that answer one business question each |

**Setup (Databricks Free Edition):**
1. **Catalog** → `workspace` → `default` → **Create → Volume** named `olist_raw`.
2. Upload the 9 Olist CSVs ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)) to the volume.
3. Import this notebook, connect to **Serverless** compute and run all cells.

The same code runs on Azure Databricks; only `RAW_PATH` changes (an ADLS path instead of a volume).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG  = "workspace"
RAW_PATH = "/Volumes/workspace/default/olist_raw"

for layer in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{layer}")

def save(df, layer, name):
    """Write a DataFrame as a Delta table, e.g. workspace.silver.orders"""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{layer}.{name}")
    print(f"saved {layer}.{name}: {spark.table(f'{CATALOG}.{layer}.{name}').count():,} rows")

def load(layer, name):
    return spark.table(f"{CATALOG}.{layer}.{name}")

print("Spark version:", spark.version)

## Bronze: raw, as it arrived

Nothing is cleaned here. Every column is read as **text** (no `inferSchema`), and two audit columns are added: `_ingested_at` (load time) and `_source_file`.
If a cleaning bug is found later, Silver can be rebuilt from Bronze without going back to the source.

- `multiLine` + `escape`: some review comments contain line breaks and quotes. Without these options one review splits into several broken rows.
- `header`: the first row holds the column names.

In [8]:
files = {
    "orders":      "olist_orders_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for name, file in files.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(f"{RAW_PATH}/{file}")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
    save(df, "bronze", name)

saved bronze.orders: 99,441 rows
saved bronze.customers: 99,441 rows
saved bronze.order_items: 112,650 rows
saved bronze.payments: 103,886 rows
saved bronze.reviews: 99,224 rows
saved bronze.products: 32,951 rows
saved bronze.sellers: 3,095 rows
saved bronze.geolocation: 1,000,163 rows
saved bronze.category_translation: 71 rows


Raw row counts: orders 99,441 · customers 99,441 · order_items 112,650 · payments 103,886 · reviews 99,224 · products 32,951 · sellers 3,095 · geolocation 1,000,163.

Reading everything as text also protects values like `customer_zip_code_prefix`: it keeps its leading zero (`01037`), whereas pandas' type inference turned it into the number `1037`. Types are cast on purpose in Silver.

In [ ]:
display(load("bronze", "customers").limit(5))

## Silver: cleaned and typed

For each table: read Bronze → cast types → apply the cleaning rules → save to Silver.
The rules were first worked out by profiling the same data in pandas, then ported to PySpark; the checks at the end confirm both give the same row counts.

### orders
Parse the 5 date columns · drop the 9 delivered orders with no carrier or customer delivery date · flag orders where the carrier date is after the customer date (23) · derive `delivery_days` and `is_late` (null when not delivered).

In [10]:
orders = load("bronze", "orders").drop("_ingested_at", "_source_file")

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders = orders.withColumn(c, F.to_timestamp(c))

# drop delivered orders with a missing carrier/customer date
bad = (F.col("order_status") == "delivered") & (
       F.col("order_delivered_carrier_date").isNull() | F.col("order_delivered_customer_date").isNull())
print("dropping:", orders.filter(bad).count())          # expected: 9
orders = orders.filter(~bad)

# flag impossible order of events (coalesce: null -> False)
orders = orders.withColumn("dt_carrier_after_customer",
    F.coalesce(F.col("order_delivered_carrier_date") > F.col("order_delivered_customer_date"), F.lit(False)))

# delivery_days: whole days between purchase and delivery (same as pandas .dt.days)
# (F.datediff would count calendar-date changes instead, so a 23:00 -> 01:00 delivery = 1 day. The pandas version used whole days too.)
orders = orders.withColumn("delivery_days",
    F.floor((F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400).cast("int"))

# is_late: only defined for delivered orders, otherwise null
orders = orders.withColumn("is_late",
    F.when(F.col("order_delivered_customer_date").isNotNull(),
           F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")))

print("carrier after customer:", orders.filter("dt_carrier_after_customer").count())   # expected: 23
save(orders, "silver", "orders")                                                        # expected: 99,432

dropping: 9
carrier after customer: 23
saved silver.orders: 99,432 rows


### customers
`customer_zip_code_prefix` is forced to a 5-character string (`lpad`). Both `customer_id` (one per order) and `customer_unique_id` (one per person) are kept: 99,441 vs 96,096.

In [ ]:
customers = load("bronze", "customers").drop("_ingested_at", "_source_file")

# Bronze read the zip as TEXT, so the leading zeros are already there.
# lpad makes it GUARANTEED (if a future file ever has "1037", it becomes "01037").
customers = customers.withColumn("customer_zip_code_prefix", F.lpad("customer_zip_code_prefix", 5, "0"))

# customer_id = one per ORDER (Olist creates a new one each time), customer_unique_id = the real person
customers.select(F.countDistinct("customer_id").alias("customer_id"),
                 F.countDistinct("customer_unique_id").alias("customer_unique_id")).show()   # 99,441 vs 96,096

save(customers, "silver", "customers")

+-----------+------------------+
|customer_id|customer_unique_id|
+-----------+------------------+
|      99441|             96096|
+-----------+------------------+

saved silver.customers: 99,441 rows


### order_items
Parse `shipping_limit_date` · cast `price`, `freight_value` and `order_item_id` · flag IQR outliers in `price_outlier` and `freight_value_outlier` (flagged, not dropped: expensive items are real sales).
Saved after the orphan step below.

In [12]:
items = load("bronze", "order_items").drop("_ingested_at", "_source_file")

items = (items
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("price",         F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double")))

def iqr_flag(df, col):
    """Add <col>_outlier = True if the value is outside Q1 - 1.5*IQR .. Q3 + 1.5*IQR. Flags, never drops."""
    q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0)   # 0 = exact quantiles; returns a plain Python list
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(f"{col}: q1={q1} q3={q3} -> keep range {low:.2f} .. {high:.2f}")
    return df.withColumn(f"{col}_outlier", (F.col(col) < low) | (F.col(col) > high))

items = iqr_flag(items, "price")
items = iqr_flag(items, "freight_value")

print("price outliers:  ", items.filter("price_outlier").count())           # expected: 8,427
print("freight outliers:", items.filter("freight_value_outlier").count())   # expected: 12,134
# saved after the orphan step below

price: q1=39.9 q3=134.9 -> keep range -102.60 .. 277.40
freight_value: q1=13.08 q3=21.15 -> keep range 0.98 .. 33.25
price outliers:   8427
freight outliers: 12134


### payments
Drop `payment_type == "not_defined"` (3 rows) · keep zero-value vouchers · set `payment_installments` 0 → 1 (2 rows) · cast numerics.

In [13]:
payments = load("bronze", "payments").drop("_ingested_at", "_source_file")

payments = (payments
    .withColumn("payment_sequential",   F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value",        F.col("payment_value").cast("double")))

# 1) "not_defined" payment type: 3 rows, no usable info -> drop
print("not_defined:", payments.filter(F.col("payment_type") == "not_defined").count())     # expected: 3
payments = payments.filter(F.col("payment_type") != "not_defined")

# 2) 0 installments is impossible (a payment is at least 1 installment) -> fix to 1
print("0 installments:", payments.filter(F.col("payment_installments") == 0).count())      # expected: 2
payments = payments.withColumn("payment_installments",
    F.when(F.col("payment_installments") == 0, 1).otherwise(F.col("payment_installments")))

# 3) zero-value vouchers are KEPT (real business event, not an error)
print("zero-value payments kept:", payments.filter(F.col("payment_value") == 0).count())
# saved after the orphan step below

not_defined: 3
0 installments: 2
zero-value payments kept: 6


### reviews
Parse both dates · cast `review_score` · add `has_comment` · keep only the **latest review per order** with a window function (`row_number` over `order_id`, newest first). 551 older reviews dropped.

In [14]:
reviews = load("bronze", "reviews").drop("_ingested_at", "_source_file")

reviews = (reviews
    .withColumn("review_creation_date",    F.to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
    .withColumn("review_score",            F.col("review_score").cast("int"))
    .withColumn("has_comment",             F.col("review_comment_message").isNotNull()))

# keep the LATEST review per order (window function, same as SQL ROW_NUMBER() OVER (...))
w = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc())
before = reviews.count()
reviews = reviews.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
print("dropped older reviews:", before - reviews.count())                                   # expected: 551

# prove it: no order has 2 reviews any more
print("orders with >1 review:", reviews.groupBy("order_id").count().filter("count > 1").count())  # expected: 0
print("rows:", reviews.count())                                                              # expected: 98,673
# saved after the orphan step below

dropped older reviews: 551
orders with >1 review: 0
rows: 98673


### products + category translation
Fix the misspelled `lenght` columns · `has_listing_info` flag and category `"unknown"` for 610 products with no listing data · weight 0 → null (4) · cast numerics · left join the English category names and fill the 3 categories missing from the translation file.

In [15]:
products = load("bronze", "products").drop("_ingested_at", "_source_file")

# 1) fix the misspelled source columns
products = (products
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length"))

# 2) cast every numeric column (loop over names instead of 7 copy-pasted lines)
for c in products.columns:
    if c.endswith(("_length", "_qty", "_g", "_cm")):
        products = products.withColumn(c, F.col(c).cast("int"))

# 3) 610 products have no name/description/category -> flag them + category "unknown"
products = products.withColumn("has_listing_info", F.col("product_name_length").isNotNull())
products = products.fillna({"product_category_name": "unknown"})

# 4) weight 0 g is impossible -> null (unknown), not 0
print("weight 0:", products.filter(F.col("product_weight_g") == 0).count())                # expected: 4
products = products.withColumn("product_weight_g",
    F.when(F.col("product_weight_g") == 0, None).otherwise(F.col("product_weight_g")))

# 5) English names: LEFT join (keep every product), then hand-fill the 3 the file doesn't cover
tr = load("bronze", "category_translation").drop("_ingested_at", "_source_file")
products = products.join(tr, "product_category_name", "left")

manual = (F.when(F.col("product_category_name") == "unknown", "unknown")
           .when(F.col("product_category_name") == "pc_gamer", "pc_gamer")
           .when(F.col("product_category_name") == "portateis_cozinha_e_preparadores_de_alimentos",
                 "portable_kitchen_food_processors"))
products = products.withColumn("product_category_name_english",
                               F.coalesce(F.col("product_category_name_english"), manual))

print("null English categories:", products.filter(F.col("product_category_name_english").isNull()).count())  # expected: 0
save(products, "silver", "products")                                                        # expected: 32,951

weight 0: 4
null English categories: 0
saved silver.products: 32,951 rows


### Orphan cascade
Dropping 9 orders leaves child rows in `order_items`, `payments` and `reviews` that point to nothing. They're counted with a `left_anti` join and removed with a `left_semi` join before saving.

In [16]:
silver_orders = load("silver", "orders")

for name, df in [("order_items", items), ("payments", payments), ("reviews", reviews)]:
    orphans = df.join(silver_orders, "order_id", "left_anti").count()    # rows whose order was dropped in C1
    print(f"{name}: {orphans} orphan rows removed")
    save(df.join(silver_orders, "order_id", "left_semi"), "silver", name)  # keep only rows WITH a parent order
# expected: order_items 112,641 · payments 103,874 · reviews 98,664

order_items: 9 orphan rows removed
saved silver.order_items: 112,641 rows
payments: 9 orphan rows removed
saved silver.payments: 103,874 rows
reviews: 9 orphan rows removed
saved silver.reviews: 98,664 rows


## Checks

The pipeline stops if any of these fail:
- Silver row counts match the pandas baseline
- primary keys are unique
- no orphan keys between tables
- types and business rules hold (review score 1-5, positive prices, `is_late` consistent with the delivery date)

In [17]:
expected = {"orders": 99_432, "customers": 99_441, "order_items": 112_641,
            "payments": 103_874, "reviews": 98_664, "products": 32_951}
for name, n in expected.items():
    got = load("silver", name).count()
    assert got == n, f"{name}: expected {n:,}, got {got:,}"

def assert_unique(name, keys):
    dups = load("silver", name).groupBy(*keys).count().filter("count > 1").count()
    assert dups == 0, f"{name}: {dups} duplicate keys on {keys}"

assert_unique("orders", ["order_id"])
assert_unique("customers", ["customer_id"])
assert_unique("order_items", ["order_id", "order_item_id"])
assert_unique("payments", ["order_id", "payment_sequential"])
assert_unique("reviews", ["order_id"])
assert_unique("products", ["product_id"])

def assert_no_orphans(child, parent, key):
    n = load("silver", child).join(load("silver", parent), key, "left_anti").count()
    assert n == 0, f"{child}: {n} rows with no matching {parent}"

assert_no_orphans("order_items", "orders", "order_id")
assert_no_orphans("payments", "orders", "order_id")
assert_no_orphans("reviews", "orders", "order_id")
assert_no_orphans("orders", "customers", "customer_id")
assert_no_orphans("order_items", "products", "product_id")

assert dict(load("silver", "orders").dtypes)["order_purchase_timestamp"] == "timestamp"
assert load("silver", "products").filter(F.col("product_category_name_english").isNull()).count() == 0

# business-rule checks
# 1) review scores must be 1-5
assert load("silver", "reviews").filter(~F.col("review_score").between(1, 5)).count() == 0, "review_score out of range"
# 2) no zero/negative price, no negative freight
assert load("silver", "order_items").filter((F.col("price") <= 0) | (F.col("freight_value") < 0)).count() == 0, "bad price/freight"
# 3) is_late is null exactly when there is no delivery date (and never null for a delivered date)
so = load("silver", "orders")
assert so.filter(F.col("order_delivered_customer_date").isNull() != F.col("is_late").isNull()).count() == 0, "is_late inconsistent"

print("all checks passed ✅")

all checks passed ✅


## Gold: one table per business question

### `gold.monthly_sales`: how are sales trending?
Delivered orders only; revenue = item prices (freight reported separately).

In [18]:
orders = load("silver", "orders").filter(F.col("order_status") == "delivered")
items  = load("silver", "order_items")

monthly = (orders.join(items, "order_id")
    .groupBy(F.date_trunc("month", "order_purchase_timestamp").cast("date").alias("month"))
    .agg(F.countDistinct("order_id").alias("n_orders"),
         F.round(F.sum("price"), 2).alias("revenue"),
         F.round(F.sum("freight_value"), 2).alias("freight"))
    .withColumn("avg_order_value", F.round(F.col("revenue") / F.col("n_orders"), 2))
    .orderBy("month"))

save(monthly, "gold", "monthly_sales")
display(monthly)        # click the chart icon above the table: line chart, x = month, y = revenue

saved gold.monthly_sales: 23 rows
+----------+--------+---------+---------+---------------+
|month     |n_orders|revenue  |freight  |avg_order_value|
+----------+--------+---------+---------+---------------+
|2016-09-01|1       |134.97   |8.49     |134.97         |
|2016-10-01|265     |40325.11 |6165.55  |152.17         |
|2016-12-01|1       |10.9     |8.72     |10.9           |
|2017-01-01|750     |111798.36|15684.01 |149.06         |
|2017-02-01|1653    |234223.4 |37015.92 |141.7          |
|2017-03-01|2546    |359198.85|55132.1  |141.08         |
|2017-04-01|2303    |340669.68|50142.72 |147.92         |
|2017-05-01|3545    |489159.25|77498.15 |137.99         |
|2017-06-01|3135    |421923.37|68127.0  |134.58         |
|2017-07-01|3872    |481604.52|84694.56 |124.38         |
|2017-08-01|4193    |554699.7 |91132.66 |132.29         |
|2017-09-01|4149    |607220.67|93662.84 |146.35         |
|2017-10-01|4478    |648247.65|102869.36|144.76         |
|2017-11-01|7288    |987648.07|165581.

### `gold.category_performance`: which categories bring in the most money, and are their customers happy?
Items sold, revenue and average review per English category, delivered orders only.

Note: reviews belong to orders, so averaging them over items weights a 5-item order 5 times. Revenue is right at item level; the star schema in notebook 02 keeps `review_score` on the order-level fact to avoid this.

In [19]:
delivered = load("silver", "orders").filter(F.col("order_status") == "delivered").select("order_id")

cat = (load("silver", "order_items")
       .join(delivered, "order_id")                                                # delivered only (inner join = filter)
       .join(load("silver", "products"), "product_id")                             # brings the English category
       .join(load("silver", "reviews").select("order_id", "review_score"), "order_id", "left")  # LEFT: not every order has a review
       .groupBy(F.col("product_category_name_english").alias("category"))
       .agg(F.count("*").alias("n_items"),
            F.round(F.sum("price"), 2).alias("revenue"),
            F.round(F.avg("review_score"), 2).alias("avg_review"))
       .orderBy(F.desc("revenue")))

save(cat, "gold", "category_performance")
display(cat.limit(10))      # expected: health_beauty first, ~1.23M

saved gold.category_performance: 74 rows
+---------------------+-------+----------+----------+
|category             |n_items|revenue   |avg_review|
+---------------------+-------+----------+----------+
|health_beauty        |9465   |1233131.72|4.19      |
|watches_gifts        |5857   |1165898.98|4.07      |
|bed_bath_table       |10953  |1023434.76|3.92      |
|sports_leisure       |8430   |954673.55 |4.17      |
|computers_accessories|7643   |888613.62 |3.99      |
|furniture_decor      |8159   |711748.69 |3.95      |
|housewares           |6795   |615628.69 |4.11      |
|cool_stuff           |3718   |610204.1  |4.2       |
|auto                 |4139   |578849.35 |4.11      |
|toys                 |4029   |471097.49 |4.21      |
+---------------------+-------+----------+----------+



### `gold.delivery_by_state`: where are deliveries slow or late?
Delivered orders, average `delivery_days` and late rate per customer state.

In [20]:
delivered = load("silver", "orders").filter(F.col("order_status") == "delivered")

state = (delivered
    .join(load("silver", "customers"), "customer_id")
    .groupBy("customer_state")
    .agg(F.count("*").alias("n_orders"),
         F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
         F.round(F.avg(F.col("is_late").cast("int")) * 100, 1).alias("late_pct"))   # avg of 1/0 = share of 1s
    .orderBy(F.desc("late_pct")))

save(state, "gold", "delivery_by_state")
print("states:", state.count())          # expected: 27
display(state)                            # expected: AL top (~24%), SP ~6%

saved gold.delivery_by_state: 27 rows
states: 27
+--------------+--------+-----------------+--------+
|customer_state|n_orders|avg_delivery_days|late_pct|
+--------------+--------+-----------------+--------+
|AL            |397     |24.0             |23.9    |
|MA            |717     |21.1             |19.7    |
|PI            |476     |19.0             |16.0    |
|CE            |1279    |20.8             |15.3    |
|SE            |335     |21.0             |15.2    |
|BA            |3256    |18.9             |14.0    |
|RJ            |12350   |14.8             |13.5    |
|TO            |274     |17.2             |12.8    |
|PA            |946     |23.3             |12.4    |
|RR            |41      |29.0             |12.2    |
|ES            |1995    |15.3             |12.2    |
|MS            |701     |15.2             |11.6    |
|PB            |517     |20.0             |11.0    |
|RN            |474     |18.8             |10.8    |
|PE            |1593    |18.0             |10.8   